# 04 Monte Carlo Risk Shape

Use cluster-level R results to inspect risk shape. Pure shuffle assumes independence. Block bootstrap keeps some consecutive-trade structure and is usually more conservative when regimes exist.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = cwd
while project_root.name != "SEN05" and project_root.parent != project_root:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

BACKTEST_ROOT = project_root / "backtest_optimize"
SINGLE_RUN_DIR = BACKTEST_ROOT / "outputs" / "single_runs"
OUTPUT_DIR = BACKTEST_ROOT / "outputs" / "monte_carlo_runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd

from backtest_optimize.analysis.monte_carlo import simulate_paths, summarize_paths

pd.set_option("display.max_columns", 80)

In [ ]:
# Option A: point this to a cluster CSV saved by notebook 01.
CLUSTER_FILE = None

# Option B: automatically use latest cluster CSV when available.
if CLUSTER_FILE is None:
    files = sorted(SINGLE_RUN_DIR.glob("*_clusters.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    CLUSTER_FILE = files[0] if files else None

if CLUSTER_FILE is None:
    raise FileNotFoundError("No cluster CSV found. Run notebook 01 first or set CLUSTER_FILE.")

clusters = pd.read_csv(CLUSTER_FILE)
r_values = clusters.loc[
    ~clusters["status"].isin(["skipped", "open_at_end"]),
    "r_result",
].dropna().astype(float).to_numpy()

print(CLUSTER_FILE)
print("R values:", len(r_values))
clusters.head()

In [ ]:
N_RUNS = 1000
RISK_PER_CLUSTER = 0.01
DRAWDOWN_THRESHOLD = 0.10

shuffle_paths = simulate_paths(r_values, n_runs=N_RUNS, method="shuffle", seed=42)
block_paths = simulate_paths(r_values, n_runs=N_RUNS, method="block_bootstrap", block_size=5, seed=42)

summary = pd.DataFrame([
    {"method": "shuffle", **summarize_paths(shuffle_paths, risk_per_cluster=RISK_PER_CLUSTER, drawdown_threshold=DRAWDOWN_THRESHOLD)},
    {"method": "block_bootstrap", **summarize_paths(block_paths, risk_per_cluster=RISK_PER_CLUSTER, drawdown_threshold=DRAWDOWN_THRESHOLD)},
])

display(summary)

In [ ]:
run_name = f"monte_carlo_{pd.Timestamp.now('UTC').strftime('%Y%m%d_%H%M%S')}"
path = OUTPUT_DIR / f"{run_name}.csv"
summary.to_csv(path, index=False)
path